# 1. Model Loading Module

This module handles loading all recipe generation models:
- GPT-2 (fine-tuned)
- Llama 3.2 1B (fine-tuned with LoRA)
- Llama 3.1 8B GGUF (quantized)

**Usage:**
```python
%run 1_model_loading.ipynb
model_dict = load_recipe_model(RecipeModelType.LLAMA_1B)
```

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2LMHeadModel, GPT2Tokenizer, BitsAndBytesConfig
from peft import PeftModel
from llama_cpp import Llama
from pathlib import Path
from typing import Dict, Any
from enum import Enum
from dataclasses import dataclass

print("✓ Imports loaded")

✓ Imports loaded


In [2]:
# Paths
PROJECT_ROOT = Path.cwd().parent.parent.parent
MODEL_DIR = PROJECT_ROOT / "models" / "recipe_generation"

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Model directory: {MODEL_DIR}")

✓ Project root: c:\Users\Champion\Documents\GitHub\cAIuldron
✓ Model directory: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation


In [3]:
class RecipeModelType(Enum):
    """Available recipe generation models"""
    GPT2 = "gpt2"
    LLAMA_1B = "llama3.2-1b"
    LLAMA_8B_GGUF = "llama3.1-8b-gguf"

@dataclass
class ModelConfig:
    name: str
    display_name: str
    speed: str
    quality: str
    vram_required: str
    description: str

MODEL_INFO = {
    RecipeModelType.GPT2: ModelConfig(
        name="gpt2-finetuned",
        display_name="GPT-2 (Fast)",
        speed="fast",
        quality="good (60-70/100)",
        vram_required="1-2 GB",
        description="Fast but lower quality"
    ),
    RecipeModelType.LLAMA_1B: ModelConfig(
        name="llama3.2-1b-finetuned",
        display_name="Llama 3.2 1B (Recommended)",
        speed="medium",
        quality="excellent (90-95/100)",
        vram_required="4-5 GB",
        description="Recommended: Fast, high quality"
    ),
    RecipeModelType.LLAMA_8B_GGUF: ModelConfig(
        name="llama3.1-8b-gguf",
        display_name="Llama 3.1 8B GGUF (Best)",
        speed="slow",
        quality="excellent (95-100/100)",
        vram_required="6 GB",
        description="Best quality but slower"
    ),
}

RECIPE_MODELS = {}
CURRENT_MODEL_TYPE = RecipeModelType.LLAMA_1B

print("✓ Model configuration loaded")
print(f"  Default: {MODEL_INFO[CURRENT_MODEL_TYPE].display_name}")

✓ Model configuration loaded
  Default: Llama 3.2 1B (Recommended)


In [4]:
def load_gpt2_model() -> Dict[str, Any]:
    """Load fine-tuned GPT-2 model"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    finetuned_dir = MODEL_DIR / "finetuned"
    
    if finetuned_dir.exists():
        tokenizer = GPT2Tokenizer.from_pretrained(str(finetuned_dir))
        tokenizer.pad_token = tokenizer.eos_token
        model = GPT2LMHeadModel.from_pretrained(str(finetuned_dir))
        model.to(device)
        model.eval()
        print(f"  ✓ Fine-tuned GPT-2 loaded to {device}")
    else:
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token
        model = GPT2LMHeadModel.from_pretrained("gpt2")
        model.to(device)
        model.eval()
        print(f"  ⚠️  Base GPT-2 loaded to {device} (fine-tuned not found)")
    
    return {"model": model, "tokenizer": tokenizer, "type": "gpt2", "device": device}

print("✓ GPT-2 loading function defined")

✓ GPT-2 loading function defined


In [5]:
def load_llama_1b_model() -> Dict[str, Any]:
    """Load Llama 3.2 1B fine-tuned model - using 4-bit quantization"""
    base_model_name = "meta-llama/Llama-3.2-1B-Instruct"
    adapter_path = MODEL_DIR / "llama3_1b_finetuned"

    # Use same 4-bit quantization as training
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"  ✓ Base model loaded (4-bit), Memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

    if adapter_path.exists():
        model = PeftModel.from_pretrained(model, str(adapter_path))
        model = model.merge_and_unload()
        print("  ✓ LoRA adapter merged - Using your fine-tuned model!")
    else:
        print("  ⚠️  Adapter not found - using base model")

    return {"model": model, "tokenizer": tokenizer, "type": "llama"}

print("✓ Llama 1B loading function defined")

✓ Llama 1B loading function defined


In [6]:
def load_llama_8b_gguf_model() -> Dict[str, Any]:
    """Load Llama 3.1 8B GGUF model - Optimized for RTX 3060 Laptop (4GB VRAM)"""
    model_path = MODEL_DIR / "Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf"
    
    llm = Llama(
        model_path=str(model_path),
        n_gpu_layers=12,        # Optimized for 4GB VRAM
        n_ctx=3072,             # Increased context for better recipes
        n_batch=256,            # Reduced to fit 4GB VRAM
        n_threads=14,           # Use all CPU cores for offloaded layers
        f16_kv=True,            # Use FP16 for KV cache (saves VRAM)
        verbose=False
    )

    print("  ✓ Llama 3.1 8B GGUF loaded (optimized for RTX 3060 Laptop 4GB)")
    return {"model": llm, "tokenizer": None, "type": "gguf"}

print("✓ Llama 8B GGUF loading function defined")

✓ Llama 8B GGUF loading function defined


In [7]:
def load_recipe_model(model_type: RecipeModelType) -> Dict[str, Any]:
    """Load specified recipe generation model (with caching)"""
    if model_type in RECIPE_MODELS:
        return RECIPE_MODELS[model_type]

    print(f"Loading {MODEL_INFO[model_type].display_name}...")

    if model_type == RecipeModelType.GPT2:
        model_dict = load_gpt2_model()
    elif model_type == RecipeModelType.LLAMA_1B:
        model_dict = load_llama_1b_model()
    elif model_type == RecipeModelType.LLAMA_8B_GGUF:
        model_dict = load_llama_8b_gguf_model()
    
    RECIPE_MODELS[model_type] = model_dict
    return model_dict

print("✓ Model loading function defined (with caching)")

✓ Model loading function defined (with caching)


## Test Model Loading

Uncomment to test:

In [8]:
model_dict = load_recipe_model(RecipeModelType.LLAMA_1B)
print(f"✓ Model loaded: {model_dict['type']}")

Loading Llama 3.2 1B (Recommended)...
  ✓ Base model loaded (4-bit), Memory: 0.94 GB


c:\Users\Champion\anaconda3\envs\pytorch_cuda\Lib\site-packages\peft\tuners\lora\bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


  ✓ LoRA adapter merged - Using your fine-tuned model!
✓ Model loaded: llama


---

**Module exports:**
- `RecipeModelType` (enum)
- `MODEL_INFO` (dict)
- `load_recipe_model()` (function)
- `RECIPE_MODELS` (cache)
- `CURRENT_MODEL_TYPE` (default model)